### Read the Bronze table

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.geolocation"

SILVER_CLEAN = "olist_mlops.silver.geolocation_clean"

SILVER_LOOKUP = "olist_mlops.silver.geolocation_zip_lookup"

SOURCE_COLUMNS = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state"
]

bronze_geo = spark.table(BRONZE_TABLE)

print("Bronze rows:", bronze_geo.count())

bronze_geo.printSchema()

### Remove exact duplicate records

In [0]:
# Select only source columns for duplicate comparison.
raw_geo = bronze_geo.select(*SOURCE_COLUMNS)

raw_count = raw_geo.count()

# Remove identical source records.
geo_deduplicated = raw_geo.dropDuplicates(SOURCE_COLUMNS)

dedup_count = geo_deduplicated.count()

duplicates_removed = raw_count - dedup_count

print("Original rows:", raw_count)
print("After deduplication:", dedup_count)
print("Duplicates removed:", duplicates_removed)

assert raw_count == 1000163
assert dedup_count == 738332
assert duplicates_removed == 261831

print("DEDUPLICATION VALIDATION PASSED")

### Clean geographic data types

- Standardize data

In [0]:
geo_clean = (
    geo_deduplicated

    # Standardize ZIP prefix
    .withColumn(
        "zip_prefix_5d",
        F.lpad(
            F.trim(F.col("geolocation_zip_code_prefix")),
            5,
            "0"
        )
    )

    # Convert latitude to DOUBLE
    .withColumn(
        "latitude",
        F.expr(
            "try_cast(geolocation_lat AS DOUBLE)"
        )
    )

    # Convert longitude to DOUBLE
    .withColumn(
        "longitude",
        F.expr(
            "try_cast(geolocation_lng AS DOUBLE)"
        )
    )

    # Standardize state abbreviation
    .withColumn(
        "geolocation_state",
        F.upper(
            F.trim(F.col("geolocation_state"))
        )
    )

    # Standardize city text
    .withColumn(
        "geolocation_city",
        F.trim(F.col("geolocation_city"))
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

geo_clean.printSchema()

- Validate coordinates and ZIP codes

In [0]:
invalid_zip = geo_clean.filter(
    F.col("zip_prefix_5d").isNull()
    | ~F.col("zip_prefix_5d").rlike(r"^[0-9]{5}$")
).count()

invalid_coordinates = geo_clean.filter(
    F.col("latitude").isNull()
    | F.col("longitude").isNull()
    | F.isnan("latitude")
    | F.isnan("longitude")
    | (F.abs(F.col("latitude")) > 90)
    | (F.abs(F.col("longitude")) > 180)
).count()

print("Invalid ZIP prefixes:", invalid_zip)
print("Invalid coordinates:", invalid_coordinates)

assert invalid_zip == 0
assert invalid_coordinates == 0

print("GEOLOCATION TYPE VALIDATION PASSED")

### Write geolocation_clean

- Save the Delta table

In [0]:
(
    geo_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_CLEAN)
)

actual_count = spark.table(SILVER_CLEAN).count()

print("Silver geolocation rows:", actual_count)

assert actual_count == 738332

print("SILVER GEOLOCATION CLEAN CREATED")

### Build the ZIP-code lookup table
- Group and calculate median coordinates

In [0]:
geo_source = spark.table(SILVER_CLEAN)

zip_lookup = (
    geo_source

    .groupBy("zip_prefix_5d")

    .agg(
        F.percentile_approx(
            "latitude", 0.5, 10000
        ).alias("latitude"),

        F.percentile_approx(
            "longitude", 0.5, 10000
        ).alias("longitude"),

        F.count("*").alias(
            "coordinate_record_count"
        ),

        F.countDistinct(
            "geolocation_city"
        ).alias("distinct_city_names"),

        F.countDistinct(
            "geolocation_state"
        ).alias("distinct_states")
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

display(zip_lookup.limit(10))

- Validate ZIP-level uniqueness

In [0]:
total_zip_records = zip_lookup.count()

duplicate_zips = (
    zip_lookup
    .groupBy("zip_prefix_5d")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

source_distinct_zips = (
    geo_source
    .select("zip_prefix_5d")
    .distinct()
    .count()
)

print("ZIP lookup records:", total_zip_records)
print("Distinct source ZIPs:", source_distinct_zips)
print("Duplicate ZIPs:", duplicate_zips)

assert total_zip_records == source_distinct_zips
assert duplicate_zips == 0

print("ZIP LOOKUP VALIDATION PASSED")

- Save the lookup

In [0]:
(
    zip_lookup.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_LOOKUP)
)

assert (
    spark.table(SILVER_LOOKUP).count()
    == total_zip_records
)

print("Created:", SILVER_LOOKUP)

### Test a real customer-to-geolocation join
- Perform a safe LEFT JOIN

In [0]:
customers = (
    spark.table("olist_mlops.silver.customers_clean")
    .select("customer_id", "customer_zip_prefix_5d")
)

geo = (
    spark.table(SILVER_LOOKUP)
    .select("zip_prefix_5d", "latitude", "longitude")
)

enriched_customers = customers.join(
    geo,
    customers.customer_zip_prefix_5d == geo.zip_prefix_5d,
    "left"
)

total_customers = customers.count()
enriched_count = enriched_customers.count()

unmatched_customers = enriched_customers.filter(
    F.col("zip_prefix_5d").isNull()
).count()

print("Original customer records:", total_customers)
print("Enriched customer records:", enriched_count)
print("Unmatched customer ZIPs:", unmatched_customers)

assert total_customers == enriched_count == 99441

print("CUSTOMER GEOLOCATION JOIN VALIDATION PASSED")

### Validate using SQL

In [0]:
%sql

SELECT
    COUNT(*) AS total_zip_records,
    COUNT(DISTINCT zip_prefix_5d) AS distinct_zip_prefixes,
    MIN(coordinate_record_count) AS min_coordinates,
    MAX(coordinate_record_count) AS max_coordinates,
    SUM(coordinate_record_count) AS total_unique_coordinates
FROM olist_mlops.silver.geolocation_zip_lookup;

In [0]:
%sql
-- investigate geographic inconsistencies
SELECT
    zip_prefix_5d,
    coordinate_record_count,
    distinct_city_names,
    distinct_states,
    latitude,
    longitude
FROM olist_mlops.silver.geolocation_zip_lookup
WHERE distinct_city_names > 1
   OR distinct_states > 1
ORDER BY distinct_states DESC,
         distinct_city_names DESC,
         coordinate_record_count DESC
LIMIT 20;